# GoodWe EV ChargeOps Assistant — Sprint 2
**EV Challenge 2026 | GoodWe Brazil**

Integrantes:
- Brenno Gomes — RM: 570525
- Eduardo Moreira — RM: 569923
- Enzo Stahal — RM: 569001
- Matheus Bruno — RM: 572944

---
### Como usar:
1. Adicione sua `OPENAI_API_KEY` nos **Secrets** do Colab (ícone 🔑)
2. Execute as células em ordem
3. Edite a variável `mensagem` na célula interativa para conversar

In [ ]:
# Instalar dependências
!pip install openai --quiet

In [ ]:
import os
import json
from datetime import datetime
from openai import OpenAI

# Carrega API Key via Colab Secrets (nunca exponha a chave no código)
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print('API Key carregada com sucesso.')
except Exception:
    print('Colab Secrets indisponivel. Configure OPENAI_API_KEY no ambiente.')

In [ ]:
SYSTEM_PROMPT = (
    'Você é o GoodWe EV ChargeOps Assistant, um assistente operacional inteligente '
    'desenvolvido para o EV Challenge 2026 da GoodWe Brazil.\n\n'
    '## SEU PAPEL\n'
    'Você auxilia síndicos, administradoras de condomínio, moradores e técnicos de '
    'manutenção a gerenciar eletropostos e sessões de recarga de veículos elétricos '
    'em condomínios residenciais equipados com carregadores GoodWe.\n\n'
    '## CONTEXTO DO PROBLEMA (EV Challenge 2026)\n'
    '1. Falta de orquestração de potência entre recargas simultâneas (risco de sobrecarga)\n'
    '2. Ausência de registro automático de sessões de recarga (kWh, duração, custo)\n'
    '3. Inexistência de relatórios e faturamento automatizado por morador\n'
    '4. Dificuldade de comunicação entre moradores e síndico sobre uso e custos\n'
    '5. Falta de diagnóstico remoto de falhas nos carregadores\n\n'
    '## SUAS COMPETÊNCIAS\n'
    '- Gestão de sessões: iniciar, pausar, encerrar e registrar recargas\n'
    '- Orquestração de potência: balanceamento entre carregadores simultâneos\n'
    '- Relatórios: consumo por morador, custo por período, histórico\n'
    '- Faturamento: cálculo de custos por kWh e tarifa vigente\n'
    '- Regras do condomínio: horários, limites, agendamentos\n'
    '- Diagnóstico de falhas: erros comuns dos carregadores GoodWe\n\n'
    '## PERSONAS QUE VOCÊ ATENDE\n'
    '- Síndico/Administrador: gestão, relatórios, cobranças, regras\n'
    '- Morador: consumo próprio, agendamento, dúvidas de uso\n'
    '- Técnico: diagnóstico, erros, especificações técnicas\n\n'
    '## COMPORTAMENTO\n'
    '- Responda sempre em português do Brasil\n'
    '- Seja objetivo e acolhedor\n'
    '- Fora do escopo, redirecione gentilmente ao tema eletropostos/GoodWe\n'
    '- Use linguagem técnica para técnicos, simples para moradores\n\n'
    '## EXEMPLOS (few-shot)\n'
    'Usuário: Quantos kWh o apt 42 consumiu em abril?\n'
    'Assistente: Acesse Relatórios > Consumo por Unidade > Abril. Serão exibidos kWh, custo e número de sessões.\n\n'
    'Usuário: Erro E04 no carregador, o que faço?\n'
    'Assistente: E04 = falha de comunicação com a rede. (1) Verifique cabo/Wi-Fi; (2) Reinicie por 10s; (3) Aguarde 2 min. Se persistir, abra chamado com código E04 e número de série.'
)

print('System prompt carregado.')

In [ ]:
class GoodWeChargeBot:
    MODEL = 'gpt-4o-mini'
    MAX_TOKENS = 1024
    TEMPERATURE = 0.4

    def __init__(self):
        api_key = os.environ.get('OPENAI_API_KEY')
        if not api_key:
            raise EnvironmentError('OPENAI_API_KEY nao encontrada.')
        self.client = OpenAI(api_key=api_key)
        self.conversation_history = []
        self.session_id = datetime.now().strftime('%Y%m%d_%H%M%S')
        print('Chatbot iniciado | Sessao:', self.session_id)

    def send_message(self, user_message):
        self.conversation_history.append({'role': 'user', 'content': user_message})
        messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + self.conversation_history
        response = self.client.chat.completions.create(
            model=self.MODEL,
            messages=messages,
            max_tokens=self.MAX_TOKENS,
            temperature=self.TEMPERATURE,
        )
        assistant_msg = response.choices[0].message.content
        self.conversation_history.append({'role': 'assistant', 'content': assistant_msg})
        return assistant_msg

    def reset(self):
        self.conversation_history = []

    def show_history(self):
        for turn in self.conversation_history:
            role = 'Voce' if turn['role'] == 'user' else 'Assistente'
            print(role + ':', turn['content'], '\n')

bot = GoodWeChargeBot()

## Célula Interativa
Edite `mensagem` e execute (Ctrl+Enter) para conversar com o chatbot.

In [ ]:
mensagem = 'Como faço para ver o consumo de energia do meu apartamento?'

print('Você:', mensagem, '\n')
resposta = bot.send_message(mensagem)
print('Assistente:', resposta)

## 5 Casos de Teste — Sprint 1

In [ ]:
test_cases = [
    'Como posso verificar o consumo de energia do meu apartamento no mes passado?',
    'Tres moradores estao carregando ao mesmo tempo e a energia caiu. O que aconteceu e como evitar?',
    'Como gero um relatorio mensal de consumo por unidade para cobrar cada morador?',
    'O carregador do box 7 esta com erro E04. O que significa e como resolver?',
    'Posso agendar a recarga para horario de tarifa mais barata? Como configuro isso?',
]

bot_test = GoodWeChargeBot()

print('=' * 60)
print('CASOS DE TESTE - GoodWe EV ChargeOps Assistant')
print('=' * 60)

for i, pergunta in enumerate(test_cases, 1):
    print('\n--- Caso', i, '---')
    print('Pergunta:', pergunta)
    bot_test.reset()
    resposta = bot_test.send_message(pergunta)
    print('Resposta:', resposta)

print('\nTodos os casos executados.')

In [ ]:
# Ver historico completo da sessao interativa
bot.show_history()